In [3]:
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)

import numpy as np
import pandas as pd
from os import listdir
from os.path import isfile, join
from mhcnames import normalize_allele_name

METHOD = "hmm_pomegranate"

# 1. Read data

In [4]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/immunogenic/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu

In [5]:
onlyfiles = [f for f in listdir(DATA_PATH) if isfile(join(DATA_PATH, f))]
ALLELES = []
per_allele_train_df = dict()
for file_name in onlyfiles:
    allele_name = file_name.split("_", 1)[1].split('.')[0]
    allele_name = normalize_allele_name(allele_name)
    per_allele_train_df[allele_name] =  pd.read_csv(DATA_PATH + file_name,sep=";")
    ALLELES.append(allele_name)
ALLELES

['HLA-A*01:01', 'HLA-A*02:01', 'HLA-A*03:01', 'HLA-B*07:02', 'HLA-B*27:05']

In [7]:
per_allele_train_df[ALLELES[1]]['immunogenic'].value_counts()

0    868
1    658
Name: immunogenic, dtype: int64

# 2. Make per length split

In [4]:
from sklearn.model_selection import KFold
MIN_LENGTH = 8
MAX_LENGTH = 30

#leave only binders
def getBindersPeptidesOnly(df):
    result_peptides = df[df['immunogenic'] == 1]['peptide'].values
    return result_peptides
def getNonBindersPeptidesOnly(df):
    result_peptides = df[df['immunogenic'] == 0]['peptide'].values
    return result_peptides

def make_per_length_dict(df, binder_flag):
    lengths = set(df['length'])
    lengths = set([length for length in lengths if length >= MIN_LENGTH and length <= MAX_LENGTH])
    per_length_dict = dict()
    for length in lengths:
        result_peptides = df[(df['immunogenic'] == binder_flag) & (df['length'] == length)]['peptide'].values
        per_length_dict[length] = result_peptides
    return per_length_dict

kf = KFold(n_splits=5, shuffle=True)
def make_per_length_per_KFold_dict(per_length_dict):

    per_length_per_kfold_peptides_train = dict()
    per_length_per_kfold_peptides_test = dict()

    print("Length \t Train/Test for splits", end="")
    for target_length in per_length_dict.keys():
        print(f"\n{target_length}:", end="\t")
        peptides_array = per_length_dict[target_length]
        per_length_per_kfold_peptides_train[target_length] = dict()
        per_length_per_kfold_peptides_test[target_length] = dict()
        if len(peptides_array) > kf.n_splits:
            i = 0
            for train_indexes_b, test_indexes_b in kf.split(peptides_array):
                print("{}/{}\t".format(len(train_indexes_b), len(test_indexes_b)), end="")
                current_train = peptides_array[train_indexes_b]
                current_test = peptides_array[test_indexes_b]
                per_length_per_kfold_peptides_train[target_length][i] = current_train
                per_length_per_kfold_peptides_test[target_length][i] = current_test
                i = i + 1
        else:

            print(f"Not enough peptides of length {target_length}", end="")
    print()
    return per_length_per_kfold_peptides_train , per_length_per_kfold_peptides_test





def addAuxColumns(per_allele_df):
    for allele_name, df in per_allele_df.items():
        df['length'] = df['peptide'].str.len()

per_allele_binders_train = dict()
per_allele_non_binders_train = dict()
per_allele_per_length_binders_train = dict()
per_allele_per_length_non_binders_train = dict()
per_allele_per_length_per_kfold_binders_train = dict()
per_allele_per_length_per_kfold_binders_test = dict()
per_allele_per_length_per_kfold_non_binders_train = dict()
per_allele_per_length_per_kfold_non_binders_test = dict()


for allele_name in ALLELES:
    print(f"Allele {allele_name}")
    df_train = per_allele_train_df[allele_name]
    df_train = df_train.sample(frac=1).reset_index(drop=True)
    per_allele_binders_train[allele_name] = getBindersPeptidesOnly(df_train)
    per_allele_non_binders_train[allele_name] = getNonBindersPeptidesOnly(df_train)
    per_allele_per_length_binders_train[allele_name] = make_per_length_dict(df_train, binder_flag=1)
    per_allele_per_length_non_binders_train[allele_name] = make_per_length_dict(df_train, binder_flag=0)
    print(f"immunogenic")
    per_allele_per_length_per_kfold_binders_train[allele_name],  per_allele_per_length_per_kfold_binders_test[allele_name] = make_per_length_per_KFold_dict(per_allele_per_length_binders_train[allele_name])
    print(f"Non-immunogenic")
    per_allele_per_length_per_kfold_non_binders_train[allele_name],  per_allele_per_length_per_kfold_non_binders_test[allele_name] = make_per_length_per_KFold_dict(per_allele_per_length_non_binders_train[allele_name])


Allele HLA-A*01:01
immunogenic
Length 	 Train/Test for splits
9:	15/4	15/4	15/4	15/4	16/3	
10:	6/2	6/2	6/2	7/1	7/1	
Non-immunogenic
Length 	 Train/Test for splits
9:	116/29	116/29	116/29	116/29	116/29	
10:	62/16	62/16	62/16	63/15	63/15	
Allele HLA-A*02:01
immunogenic
Length 	 Train/Test for splits
8:	6/2	6/2	6/2	7/1	7/1	
9:	398/100	398/100	398/100	399/99	399/99	
10:	121/31	121/31	122/30	122/30	122/30	
11:	Not enough peptides of length 11
Non-immunogenic
Length 	 Train/Test for splits
8:	4/2	5/1	5/1	5/1	5/1	
9:	352/88	352/88	352/88	352/88	352/88	
10:	278/70	278/70	278/70	279/69	279/69	
11:	59/15	59/15	59/15	59/15	60/14	
Allele HLA-A*03:01
immunogenic
Length 	 Train/Test for splits
8:	Not enough peptides of length 8
9:	34/9	34/9	34/9	35/8	35/8	
10:	20/6	21/5	21/5	21/5	21/5	
11:	Not enough peptides of length 11
12:	Not enough peptides of length 12
Non-immunogenic
Length 	 Train/Test for splits
8:	Not enough peptides of length 8
9:	68/18	69/17	69/17	69/17	69/17	
10:	18/5	18/5	18/5	19/4	19/

# 3. Save per_length_per_split to files

In [5]:
import os
from collections import defaultdict

TARGET_PATH_TO_FILES = DATA_PATH  + '/per_length_per_kfold_split/'
if not os.path.exists(TARGET_PATH_TO_FILES):
    os.makedirs(TARGET_PATH_TO_FILES)

def save_data(per_allele_per_length_per_kfold_data, data_type=None, dataset_type=None):
    assert data_type in ["immunogenic", "nonimmunogenic"]
    assert dataset_type in ["train", "test"]

    per_allele_dfs = defaultdict(list)
    for allele_name in per_allele_per_length_per_kfold_data.keys():
        for length, per_k_fold_dict in per_allele_per_length_per_kfold_data[allele_name].items():
                for split_num, peptides in per_k_fold_dict.items():
                    df = pd.DataFrame({"peptide":peptides, "length":length, "split": split_num})
                    per_allele_dfs[allele_name].append(df)
        total_df= pd.concat(per_allele_dfs[allele_name])
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
        total_df.to_csv(f'{TARGET_PATH_TO_FILES}{target_allele_name}_{data_type}_{dataset_type}.csv',sep=";", index=False)


save_data(per_allele_per_length_per_kfold_binders_train, "immunogenic", "train")
save_data(per_allele_per_length_per_kfold_non_binders_train, "nonimmunogenic", "train")
save_data(per_allele_per_length_per_kfold_binders_test, "immunogenic", "test")
save_data(per_allele_per_length_per_kfold_non_binders_test, "nonimmunogenic", "test")